In [1]:
import logging

from marmopose.version import __version__ as marmopose_version
from marmopose.config import Config
from marmopose.processing.prediction import Predictor
from marmopose.visualization.display_2d import Visualizer2D
from marmopose.visualization.display_3d import Visualizer3D
from marmopose.processing.triangulation import Reconstructor3D

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(name)s - %(message)s')
logger = logging.getLogger(__name__)

logger.info(f'MarmoPose version: {marmopose_version}')

2026-05-25 19:48:40,670 - INFO - __main__ - MarmoPose version: 1.2.2


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [ ]:
# Path to the configuration file
config_path = 'E:/KAIST/Marmopose/MarmoPose/configs/default_6pt.yaml'
# config_path = 'C:/Users/issac/Documents/MarmoPose/configs/default.yaml'

# Configuration setup
# You can either modify the parameters directly in the YAML configuration file (default.yaml),
# or override specific parameters directly in the code below, as shown.
config = Config(
    config_path=config_path,
    
    # The following parameters override those in the default.yaml file
    n_tracks=2, 

    project='E:/KAIST/Marmopose/MarmoPose/demos/pair_6pt',
    det_model='E:/KAIST/Marmopose/MarmoPose/models/detection_model',
    pose_model='E:/KAIST/Marmopose/MarmoPose/models/pose_model',
    dae_model='E:/KAIST/Marmopose/MarmoPose/models/dae_model',

    dae_enable=False,
    do_optimize=True
)

2026-05-25 19:48:59,841 - INFO - marmopose.config - *** Overriding config *** | n_tracks: 2 -> 2
2026-05-25 19:48:59,842 - INFO - marmopose.config - *** Overriding config *** | project: ../demos/pair_6pt -> E:/KAIST/Marmopose/MarmoPose/demos/pair_6pt
2026-05-25 19:48:59,843 - INFO - marmopose.config - *** Overriding config *** | det_model: ../models/detection_model -> E:/KAIST/Marmopose/MarmoPose/models/detection_model
2026-05-25 19:48:59,843 - INFO - marmopose.config - *** Overriding config *** | pose_model: ../models/pose_model -> E:/KAIST/Marmopose/MarmoPose/models/pose_model
2026-05-25 19:48:59,845 - INFO - marmopose.config - *** Overriding config *** | dae_model: ../models/dae_model -> E:/KAIST/Marmopose/MarmoPose/models/dae_model
2026-05-25 19:48:59,846 - INFO - marmopose.config - *** Overriding config *** | dae_enable: False -> True
2026-05-25 19:48:59,847 - INFO - marmopose.config - *** Overriding config *** | do_optimize: True -> True


In [ ]:
# Run 2D prediction
# predictor = Predictor(config, batch_size=4)
# predictor.predict()

In [3]:
from pathlib import Path

project_dir = Path('E:/KAIST/Marmopose/MarmoPose/demos/pair_6pt')

for file in [
    project_dir / "points_3d" / "original.h5",
    project_dir / "points_3d" / "optimized.h5",
]:
    if file.exists():
        print("Deleting old 3D cache:", file)
        file.unlink()
    else:
        print("No old cache found:", file)

No old cache found: E:\KAIST\Marmopose\MarmoPose\demos\pair_6pt\points_3d\original.h5
No old cache found: E:\KAIST\Marmopose\MarmoPose\demos\pair_6pt\points_3d\optimized.h5


In [4]:
# Make sure that it's using the 2d pose estimate I replaced with (using DLC) and not from the original demo, SLEAP:
from pathlib import Path

reconstructor_3d = Reconstructor3D(config)

print("2D file being read:")
print(reconstructor_3d.points_2d_path)
print("Exists:", Path(reconstructor_3d.points_2d_path).exists())

print("\n3D output will be saved to:")
print(reconstructor_3d.points_3d_path)
print(reconstructor_3d.points_3d_optimized_path)

2026-05-25 19:49:06,351 - INFO - marmopose.processing.triangulation - Loaded camera group from: E:\KAIST\Marmopose\MarmoPose\demos\pair_6pt\calibration\camera_params.json
2026-05-25 19:49:06,354 - INFO - marmopose.processing.triangulation - Loaded DAE from: E:\KAIST\Marmopose\MarmoPose\models\dae_model\best.pth
2026-05-25 19:49:06,430 - INFO - marmopose.processing.autoencoder - DAE device: cuda
2026-05-25 19:49:06,431 - INFO - marmopose.processing.triangulation - Enabled filling of missing values using Denoising Autoencoder


2D file being read:
E:\KAIST\Marmopose\MarmoPose\demos\pair_6pt\points_2d\original.h5
Exists: True

3D output will be saved to:
E:\KAIST\Marmopose\MarmoPose\demos\pair_6pt\points_3d\original.h5
E:\KAIST\Marmopose\MarmoPose\demos\pair_6pt\points_3d\optimized.h5


In [5]:
# Run 3D triangulation and (optional) optimization 
config.optimization["bodypart_distance_weak"] = {}

reconstructor_3d = Reconstructor3D(config)
reconstructor_3d.triangulate()

2026-05-25 19:49:10,548 - INFO - marmopose.processing.triangulation - Loaded camera group from: E:\KAIST\Marmopose\MarmoPose\demos\pair_6pt\calibration\camera_params.json
2026-05-25 19:49:10,549 - INFO - marmopose.processing.triangulation - Loaded DAE from: E:\KAIST\Marmopose\MarmoPose\models\dae_model\best.pth
2026-05-25 19:49:10,552 - INFO - marmopose.processing.autoencoder - DAE device: cuda
2026-05-25 19:49:10,553 - INFO - marmopose.processing.triangulation - Enabled filling of missing values using Denoising Autoencoder
2026-05-25 19:49:10,556 - INFO - marmopose.utils.data_io - Loaded 2D points and bboxes from E:\KAIST\Marmopose\MarmoPose\demos\pair_6pt\points_2d\original.h5 with order: ['bak-1-2', 'bak-2-2', 'bak-3-2', 'bak-4-2']
Triangulating... : 100%|█████████████████████████████████████| 525/525 [00:03<00:00, 133.76frames/s]
2026-05-25 19:49:14,534 - INFO - marmopose.utils.data_io - Saving 3D points for track1 in E:\KAIST\Marmopose\MarmoPose\demos\pair_6pt\points_3d\original.h

RuntimeError: mat1 and mat2 shapes cannot be multiplied (525x18 and 48x128)

In [ ]:
# Check output
from pathlib import Path

project_dir = Path('E:/KAIST/Marmopose/MarmoPose/demos/pair_6pt')

print("3D original exists:", (project_dir / "points_3d" / "original.h5").exists())
print("3D optimized exists:", (project_dir / "points_3d" / "optimized.h5").exists())

In [ ]:
# Visualize 2D results if needed
visualizer_2d = Visualizer2D(config)
visualizer_2d.generate_videos_2d()

In [ ]:
# Visualize 3D results if needed
visualizer_3d = Visualizer3D(config)

# source_3d: 'original' indicates the 3D poses before optimization, 'optimized' indicates the 3D poses after optimization
# video_type: 'composite' indicates the 3D video is composited with 2D video, '3d' indicates only 3D video is shown; generating 'composite' video is about 2x slower than just '3d' video
visualizer_3d.generate_video_3d(source_3d='optimized', start_frame_idx=0, end_frame_idx=None, video_type='composite')

# visualizer_3d.generate_video_3d(source_3d='original', video_type='composite')
# visualizer_3d.generate_video_3d(source_3d='original', video_type='3d')
# visualizer_3d.generate_video_3d(source_3d='optimized', video_type='3d')